In [16]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.tree import DecisionTreeClassifier, export_text

# 1. Tải dữ liệu
df = yf.download('FPT.VN', start='2023-01-01', end='2026-01-01')
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# 2. Xây dựng các đặc trưng cốt lõi
df['Return_5d'] = df['Close'].pct_change(5)
df['SMA50'] = df['Close'].rolling(window=50).mean()
df['Dist_SMA50'] = (df['Close'] - df['SMA50']) / df['SMA50']

df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# 3. Chia tập dữ liệu (70% Train - 30% Test)
train_size = int(len(df) * 0.7)
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:].copy()

features = ['Return_5d', 'Dist_SMA50']
X_train = train_df[features]
y_train = train_df['Target']

# 4. Ép mô hình cực kỳ đơn giản (max_depth = 2) để tránh học vẹt
ml_model = DecisionTreeClassifier(max_depth=2, random_state=42)
ml_model.fit(X_train, y_train)

print("--- LÝ THUYẾT TINH GỌN (CÔ ĐỌNG BỞI ML) ---")
print(export_text(ml_model, feature_names=features))

# 5. Kiểm định trên tập Test
X_test = test_df[features]
test_df['ML_Signal'] = ml_model.predict(X_test)

test_df['Market_Return'] = test_df['Close'].pct_change()
test_df['Strategy_Return'] = test_df['ML_Signal'].shift(1) * test_df['Market_Return']

strategy_return = (1 + test_df['Strategy_Return'].dropna()).prod() - 1
market_return = (1 + test_df['Market_Return'].dropna()).prod() - 1

print(f"\n--- KẾT QUẢ KIỂM ĐỊNH VỚI LÝ THUYẾT TINH GỌN ---")
print(f"Lợi nhuận Thị trường (Buy & Hold): {market_return * 100:.2f}%")
print(f"Lợi nhuận theo Lý thuyết Tinh gọn: {strategy_return * 100:.2f}%")

YF.download() has changed argument auto_adjust default to True
[*********************100%***********************]  1 of 1 completed

--- LÝ THUYẾT TINH GỌN (CÔ ĐỌNG BỞI ML) ---
|--- Return_5d <= -0.05
|   |--- Dist_SMA50 <= -0.02
|   |   |--- class: 1
|   |--- Dist_SMA50 >  -0.02
|   |   |--- class: 1
|--- Return_5d >  -0.05
|   |--- Return_5d <= -0.04
|   |   |--- class: 0
|   |--- Return_5d >  -0.04
|   |   |--- class: 0


--- KẾT QUẢ KIỂM ĐỊNH VỚI LÝ THUYẾT TINH GỌN ---
Lợi nhuận Thị trường (Buy & Hold): -20.81%
Lợi nhuận theo Lý thuyết Tinh gọn: 5.65%


In [17]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.tree import DecisionTreeClassifier, export_text

# 1. Tải dữ liệu HPG từ 2023 đến 2026
df = yf.download('HPG.VN', start='2023-01-01', end='2026-01-01')
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# 2. Xây dựng các đặc trưng cốt lõi (Features)
df['Return_5d'] = df['Close'].pct_change(5)
df['SMA50'] = df['Close'].rolling(window=50).mean()
df['Dist_SMA50'] = (df['Close'] - df['SMA50']) / df['SMA50']

# Biến mục tiêu: Ngày mai giá tăng (1) hay giảm (0)
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# 3. Chia tập dữ liệu (70% Train - 30% Test)
train_size = int(len(df) * 0.7)
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:].copy()

features = ['Return_5d', 'Dist_SMA50']
X_train = train_df[features]
y_train = train_df['Target']

# 4. Huấn luyện mô hình với cấu trúc tinh gọn (max_depth = 2)
ml_model = DecisionTreeClassifier(max_depth=2, random_state=42)
ml_model.fit(X_train, y_train)

print("--- LÝ THUYẾT KHAI PHÁ TRÊN HPG ---")
print(export_text(ml_model, feature_names=features))

# 5. Kiểm định hiệu suất trên tập Test của HPG
X_test = test_df[features]
test_df['ML_Signal'] = ml_model.predict(X_test)

test_df['Market_Return'] = test_df['Close'].pct_change()
test_df['Strategy_Return'] = test_df['ML_Signal'].shift(1) * test_df['Market_Return']

strategy_return = (1 + test_df['Strategy_Return'].dropna()).prod() - 1
market_return = (1 + test_df['Market_Return'].dropna()).prod() - 1

print(f"\n--- KẾT QUẢ KIỂM ĐỊNH TRÊN HPG.VN ---")
print(f"Lợi nhuận Thị trường (Buy & Hold HPG): {market_return * 100:.2f}%")
print(f"Lợi nhuận theo Lý thuyết Machine Learning: {strategy_return * 100:.2f}%")

YF.download() has changed argument auto_adjust default to True
[*********************100%***********************]  1 of 1 completed

--- LÝ THUYẾT KHAI PHÁ TRÊN HPG ---
|--- Return_5d <= -0.02
|   |--- Dist_SMA50 <= -0.03
|   |   |--- class: 1
|   |--- Dist_SMA50 >  -0.03
|   |   |--- class: 0
|--- Return_5d >  -0.02
|   |--- Dist_SMA50 <= -0.04
|   |   |--- class: 0
|   |--- Dist_SMA50 >  -0.04
|   |   |--- class: 0


--- KẾT QUẢ KIỂM ĐỊNH TRÊN HPG.VN ---
Lợi nhuận Thị trường (Buy & Hold HPG): 14.16%
Lợi nhuận theo Lý thuyết Machine Learning: 0.61%


In [18]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.tree import DecisionTreeClassifier, export_text

# 1. Tải dữ liệu
df = yf.download('HPG.VN', start='2023-01-01', end='2026-01-01')
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# 2. Xây dựng không gian đặc trưng mở rộng (Features)
df['Return_5d'] = df['Close'].pct_change(5)
df['SMA50'] = df['Close'].rolling(window=50).mean()
df['Dist_SMA50'] = (df['Close'] - df['SMA50']) / df['SMA50']

# Thêm chỉ báo động lượng RSI (14 kỳ)
delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

# Thêm chỉ báo độ dốc xu hướng SMA50 (để biết xu hướng đang hướng lên hay xuống)
df['SMA_Slope'] = df['SMA50'].pct_change(5)

# Biến mục tiêu: Ngày mai giá tăng (1) hay giảm (0)
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# 3. Chia tập dữ liệu (70% Train - 30% Test)
train_size = int(len(df) * 0.7)
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:].copy()

features = ['Return_5d', 'Dist_SMA50', 'RSI', 'SMA_Slope']
X_train = train_df[features]
y_train = train_df['Target']

# 4. Huấn luyện mô hình với max_depth = 3 để có đủ không gian kết hợp cả Tấn công và Phòng thủ
ml_model = DecisionTreeClassifier(max_depth=3, random_state=42)
ml_model.fit(X_train, y_train)

print("--- LÝ THUYẾT ĐA CHẾ ĐỘ (MỞ RỘNG ĐẶC TRƯNG) ---")
print(export_text(ml_model, feature_names=features))

# 5. Kiểm định hiệu suất trên tập Test
X_test = test_df[features]
test_df['ML_Signal'] = ml_model.predict(X_test)

test_df['Market_Return'] = test_df['Close'].pct_change()
test_df['Strategy_Return'] = test_df['ML_Signal'].shift(1) * test_df['Market_Return']

strategy_return = (1 + test_df['Strategy_Return'].dropna()).prod() - 1
market_return = (1 + test_df['Market_Return'].dropna()).prod() - 1

print(f"\n--- KẾT QUẢ KIỂM ĐỊNH MÔ HÌNH MỚI TRÊN HPG ---")
print(f"Lợi nhuận Thị trường (Buy & Hold HPG): {market_return * 100:.2f}%")
print(f"Lợi nhuận theo Mô hình Đa chế độ: {strategy_return * 100:.2f}%")

YF.download() has changed argument auto_adjust default to True
[*********************100%***********************]  1 of 1 completed

--- LÝ THUYẾT ĐA CHẾ ĐỘ (MỞ RỘNG ĐẶC TRƯNG) ---
|--- RSI <= 41.12
|   |--- SMA_Slope <= -0.01
|   |   |--- SMA_Slope <= -0.01
|   |   |   |--- class: 0
|   |   |--- SMA_Slope >  -0.01
|   |   |   |--- class: 1
|   |--- SMA_Slope >  -0.01
|   |   |--- Dist_SMA50 <= -0.02
|   |   |   |--- class: 0
|   |   |--- Dist_SMA50 >  -0.02
|   |   |   |--- class: 1
|--- RSI >  41.12
|   |--- Dist_SMA50 <= -0.04
|   |   |--- class: 0
|   |--- Dist_SMA50 >  -0.04
|   |   |--- SMA_Slope <= -0.00
|   |   |   |--- class: 1
|   |   |--- SMA_Slope >  -0.00
|   |   |   |--- class: 0


--- KẾT QUẢ KIỂM ĐỊNH MÔ HÌNH MỚI TRÊN HPG ---
Lợi nhuận Thị trường (Buy & Hold HPG): 13.14%
Lợi nhuận theo Mô hình Đa chế độ: 25.36%


In [19]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.tree import DecisionTreeClassifier, export_text

# 1. Tải dữ liệu VCB từ 2023 đến 2026
df = yf.download('VCB.VN', start='2023-01-01', end='2026-01-01')
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# 2. Xây dựng không gian đặc trưng (Features) đa chế độ
df['Return_5d'] = df['Close'].pct_change(5)
df['SMA50'] = df['Close'].rolling(window=50).mean()
df['Dist_SMA50'] = (df['Close'] - df['SMA50']) / df['SMA50']

# Chỉ báo RSI (14 kỳ)
delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

# Độ dốc xu hướng SMA50
df['SMA_Slope'] = df['SMA50'].pct_change(5)

# Biến mục tiêu: Ngày mai giá tăng (1) hay giảm (0)
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# 3. Chia tập dữ liệu (70% Train - 30% Test)
train_size = int(len(df) * 0.7)
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:].copy()

features = ['Return_5d', 'Dist_SMA50', 'RSI', 'SMA_Slope']
X_train = train_df[features]
y_train = train_df['Target']

# 4. Huấn luyện mô hình với max_depth = 3
ml_model = DecisionTreeClassifier(max_depth=3, random_state=42)
ml_model.fit(X_train, y_train)

print("--- LÝ THUYẾT ĐA CHẾ ĐỘ ĐƯỢC HỌC TỪ VCB ---")
print(export_text(ml_model, feature_names=features))

# 5. Kiểm định hiệu suất trên tập Test của VCB
X_test = test_df[features]
test_df['ML_Signal'] = ml_model.predict(X_test)

test_df['Market_Return'] = test_df['Close'].pct_change()
test_df['Strategy_Return'] = test_df['ML_Signal'].shift(1) * test_df['Market_Return']

strategy_return = (1 + test_df['Strategy_Return'].dropna()).prod() - 1
market_return = (1 + test_df['Market_Return'].dropna()).prod() - 1

print(f"\n--- KẾT QUẢ KIỂM ĐỊNH MÔ HÌNH TRÊN VCB.VN ---")
print(f"Lợi nhuận Thị trường (Buy & Hold VCB): {market_return * 100:.2f}%")
print(f"Lợi nhuận theo Mô hình Đa chế độ: {strategy_return * 100:.2f}%")

YF.download() has changed argument auto_adjust default to True
[*********************100%***********************]  1 of 1 completed

--- LÝ THUYẾT ĐA CHẾ ĐỘ ĐƯỢC HỌC TỪ VCB ---
|--- RSI <= 51.48
|   |--- SMA_Slope <= -0.00
|   |   |--- RSI <= 42.58
|   |   |   |--- class: 0
|   |   |--- RSI >  42.58
|   |   |   |--- class: 1
|   |--- SMA_Slope >  -0.00
|   |   |--- Dist_SMA50 <= 0.05
|   |   |   |--- class: 1
|   |   |--- Dist_SMA50 >  0.05
|   |   |   |--- class: 0
|--- RSI >  51.48
|   |--- SMA_Slope <= 0.01
|   |   |--- SMA_Slope <= 0.00
|   |   |   |--- class: 0
|   |   |--- SMA_Slope >  0.00
|   |   |   |--- class: 0
|   |--- SMA_Slope >  0.01
|   |   |--- SMA_Slope <= 0.01
|   |   |   |--- class: 1
|   |   |--- SMA_Slope >  0.01
|   |   |   |--- class: 0


--- KẾT QUẢ KIỂM ĐỊNH MÔ HÌNH TRÊN VCB.VN ---
Lợi nhuận Thị trường (Buy & Hold VCB): -7.20%
Lợi nhuận theo Mô hình Đa chế độ: -6.35%


In [21]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.tree import DecisionTreeClassifier

# 1. Định nghĩa rổ tài sản (Universe)
tickers = ['FPT.VN', 'HPG.VN', 'VCB.VN', 'MWG.VN']
start_date = '2023-01-01'
end_date = '2026-01-01'

# Tải dữ liệu toàn bộ rổ
data = {}
for ticker in tickers:
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)
  if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)
  data[ticker] = df

# Tạo DataFrame chứa giá Close và Return của từng mã
close_df = pd.DataFrame(
    {ticker: df['Close'] for ticker, d in data.items() if not (df := d).empty}
)
close_df = close_df.dropna()

# 2. Xây dựng tín hiệu giao dịch cho từng mã dựa trên Mô hình Đa chế độ
signals_df = pd.DataFrame(index=close_df.index)
returns_df = close_df.pct_change()

for ticker in tickers:
  df = data[ticker].copy()

  # Feature Engineering
  df['Return_5d'] = df['Close'].pct_change(5)
  df['SMA50'] = df['Close'].rolling(window=50).mean()
  df['Dist_SMA50'] = (df['Close'] - df['SMA50']) / df['SMA50']

  delta = df['Close'].diff()
  gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
  loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
  rs = gain / loss
  df['RSI'] = 100 - (100 / (1 + rs))
  df['SMA_Slope'] = df['SMA50'].pct_change(5)

  df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
  df = df.replace([np.inf, -np.inf], np.nan).dropna()

  if len(df) > 100:
    train_size = int(len(df) * 0.7)
    train_df = df.iloc[:train_size]

    features = ['Return_5d', 'Dist_SMA50', 'RSI', 'SMA_Slope']
    X_train = train_df[features]
    y_train = train_df['Target']

    # Huấn luyện mô hình
    ml_model = DecisionTreeClassifier(max_depth=3, random_state=42)
    ml_model.fit(X_train, y_train)

    # Dự báo trên toàn bộ dữ liệu của mã đó và chuyển thành Series theo index của df
    X_all = df[features]
    preds = ml_model.predict(X_all)

    # Đưa về Pandas Series gắn đúng index ngày tháng, sau đó đồng bộ với close_df
    pred_series = pd.Series(preds, index=df.index)
    signals_df[ticker] = (
        pred_series.reindex(close_df.index).ffill().fillna(0)
    )

# 3. Xây dựng Hiệu suất Danh mục Đa tài sản
position_df = signals_df.shift(1)

# Lợi nhuận hàng ngày của chiến lược danh mục (Trung bình các mã có tín hiệu mua)
strategy_daily_returns = (position_df * returns_df).mean(axis=1)
market_daily_returns = returns_df.mean(axis=1)  # Đại diện benchmark rổ

# Tính tổng hiệu suất tích lũy
portfolio_total_return = (1 + strategy_daily_returns.dropna()).prod() - 1
market_total_return = (1 + market_daily_returns.dropna()).prod() - 1

print(f'--- KẾT QUẢ BACKTEST DANH MỤC ĐA TÀI SẢN ---')
print(f'Lợi nhuận Trung bình Rổ (Buy & Hold Market): {market_total_return * 100:.2f}%')
print(
    f'Lợi nhuận Hệ thống Quét Đa tài sản (ML Strategy): {portfolio_total_return * 100:.2f}%'
)

YF.download() has changed argument auto_adjust default to True
YF.download() has changed argument auto_adjust default to True
YF.download() has changed argument auto_adjust default to True
YF.download() has changed argument auto_adjust default to True


--- KẾT QUẢ BACKTEST DANH MỤC ĐA TÀI SẢN ---
Lợi nhuận Trung bình Rổ (Buy & Hold Market): 73.53%
Lợi nhuận Hệ thống Quét Đa tài sản (ML Strategy): 93.80%


In [22]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.tree import DecisionTreeClassifier

# 1. Định nghĩa rổ tài sản (Universe)
tickers = ['FPT.VN', 'HPG.VN', 'VCB.VN', 'MWG.VN']
start_date = '2023-01-01'
end_date = '2026-01-01'

# Tải dữ liệu toàn bộ rổ
data = {}
for ticker in tickers:
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)
  if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)
  data[ticker] = df

close_df = pd.DataFrame(
    {ticker: df['Close'] for ticker, d in data.items() if not (df := d).empty}
)
close_df = close_df.dropna()
returns_df = close_df.pct_change()

# 2. Xây dựng tín hiệu giao dịch cho từng mã
signals_df = pd.DataFrame(index=close_df.index)

for ticker in tickers:
  df = data[ticker].copy()
  
  # Feature Engineering
  df['Return_5d'] = df['Close'].pct_change(5)
  df['SMA50'] = df['Close'].rolling(window=50).mean()
  df['Dist_SMA50'] = (df['Close'] - df['SMA50']) / df['SMA50']
  
  delta = df['Close'].diff()
  gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
  loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
  rs = gain / loss
  df['RSI'] = 100 - (100 / (1 + rs))
  df['SMA_Slope'] = df['SMA50'].pct_change(5)
  
  df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
  df = df.replace([np.inf, -np.inf], np.nan).dropna()
  
  if len(df) > 100:
    train_size = int(len(df) * 0.7)
    train_df = df.iloc[:train_size]
    
    features = ['Return_5d', 'Dist_SMA50', 'RSI', 'SMA_Slope']
    X_train = train_df[features]
    y_train = train_df['Target']
    
    ml_model = DecisionTreeClassifier(max_depth=3, random_state=42)
    ml_model.fit(X_train, y_train)
    
    X_all = df[features]
    preds = ml_model.predict(X_all)
    pred_series = pd.Series(preds, index=df.index)
    signals_df[ticker] = pred_series.reindex(close_df.index).ffill().fillna(0)

# 3. Phân bổ tỷ trọng thông minh (Volatility Targeting & Risk Parity)
# Tính độ biến động cuộn 20 ngày của từng mã
rolling_vol = returns_df.rolling(window=20).std()

# Tỷ trọng thô = Tín hiệu ML * (1 / Độ biến động) -> Rủi ro cao phân bổ ít, rủi ro thấp phân bổ nhiều
raw_weights = pd.DataFrame(0.0, index=close_df.index, columns=tickers)
for ticker in tickers:
    inv_vol = 1.0 / rolling_vol[ticker]
    raw_weights[ticker] = signals_df[ticker] * inv_vol

# Chuẩn hóa tổng tỷ trọng mỗi ngày về 1 (hoặc nhỏ hơn nếu không có tín hiệu mua)
row_sums = raw_weights.sum(axis=1)
weights_df = raw_weights.div(row_sums, axis=0).fillna(0.0)

# Dịch chuyển trọng số 1 ngày để tránh look-ahead bias
portfolio_weights = weights_df.shift(1).fillna(0.0)

# 4. Tính toán Hiệu suất có trừ Chi phí Giao dịch (Transaction Costs)
portfolio_daily_returns = (portfolio_weights * returns_df).sum(axis=1)

# Tính mức độ xoay vòng danh mục (Turnover) để tính phí
turnover = (portfolio_weights - portfolio_weights.shift(1).fillna(0)).abs().sum(axis=1)
tx_cost_rate = 0.0015  # 0.15% phí giao dịch + thuế mỗi lần thay đổi trạng thái
transaction_costs = turnover * tx_cost_rate

net_strategy_returns = portfolio_daily_returns - transaction_costs

# 5. Tổng kết kết quả
portfolio_total_return = (1 + net_strategy_returns.dropna()).prod() - 1
market_total_return = (1 + returns_df.mean(axis=1).dropna()).prod() - 1

print(f"--- KẾT QUẢ QUỸ ĐỊNH LƯỢNG CHUYÊN NGHIỆP ---")
print(f"Lợi nhuận Thị trường (Buy & Hold Market): {market_total_return * 100:.2f}%")
print(f"Lợi nhuận Hệ thống Quỹ (Đã trừ phí giao dịch & Volatility Targeting): {portfolio_total_return * 100:.2f}%")

YF.download() has changed argument auto_adjust default to True
YF.download() has changed argument auto_adjust default to True
YF.download() has changed argument auto_adjust default to True
YF.download() has changed argument auto_adjust default to True


--- KẾT QUẢ QUỸ ĐỊNH LƯỢNG CHUYÊN NGHIỆP ---
Lợi nhuận Thị trường (Buy & Hold Market): 73.53%
Lợi nhuận Hệ thống Quỹ (Đã trừ phí giao dịch & Volatility Targeting): 195.61%
